In [1]:
import sys
print(sys.executable)

C:\Users\felip\AppData\Local\Programs\Python\Python312\python.exe


In [2]:
import sys
import os
import pandas as pd
import numpy as np
import torch
import time

sys.path.append(os.path.abspath('../src'))

from sklearn.model_selection import train_test_split
from sklearn.base import clone
from DataLoader import DataLoader
from DataSplitter import DataSplitter
from PreProcessorDL import PreProcessorDL
from Vectorizer import Vectorizer, Word2VecEmbedding
from Trainer import Trainer
from ModelCollection import ModelCollection
from PipelineDL import PipelineDL
from CrossValidation import CrossValidation

In [3]:
n_threads = os.cpu_count()
torch.set_num_threads(n_threads)

In [4]:
## Loading data
data_loader = DataLoader(path_train="../data/labeledTrainData.csv", path_test="../data/testData.csv")
data_splitter = DataSplitter(target_column="sentiment")
train, test = data_loader.load()
train.shape, test.shape

((25000, 3), (25000, 2))

In [5]:
## splitting data
X, y = train.drop(columns=['sentiment']), train['sentiment']
print(X.shape, y.shape)
X_train, X_test, y_train, y_test = data_splitter.split(train)
X_train.shape, X_test.shape, y_train.shape, y_test.shape, X_train.shape[0] / (X_test.shape[0] + X_train.shape[0])

(25000, 2) (25000,)


((20000, 2), (5000, 2), (20000,), (5000,), 0.8)

In [6]:
# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Multi-Layer Perceptron (MLP)

In [7]:
# DL Pipeline 
params = {'vectorizer_method': "tf-idf", 'model_name': "MLP", 'epochs': 20, 'batch_size': 256}
params['device'] = "cpu"

pipeline = PipelineDL(**params)
pipeline

,vectorizer_method,'tf-idf'
,model_name,'MLP'
,epochs,20
,batch_size,256
,device,'cpu'
,preprocessor_type,'default'
,lowercase,None
,remove_punctuation,None
,embedding_dim,None
,window,None
,min_count,None


In [8]:
# Cross-Validation
cv = CrossValidation(pipeline=clone(pipeline))

# Holdout Validation
#cv_score = cv.evaluate_one_fold(X_train, y_train, X_test, y_test)
#print("Holdout Validation score: ", cv_score)

## K-fold Cross-Validation 
#cv_score = cv.evaluate(X, y)
#print("K-fold CV score: ", cv_score)

In [9]:
## Hyper-parameter tunning
param_grid = {
    "linear_layer_sizes": [(128,), (128,64), (128,64,32)],
    "lr": [1e-4, 1e-3, 1e-2],
    "vectorizer_method": ["tf-idf", "bag of words"]
}

#cv_mlp_results, loss_curves, _ = cv.hyper_param_tune_one_fold(X_train, y_train, X_test, y_test, param_grid, 
#                                                           reduced=True, train_size=2000, return_loss_curves=True)
#cv_mlp_results

In [10]:
## Create submission using best hyper-parameters
#preds = cv.pipeline.fit(X, y).predict(test)
#submission = pd.DataFrame({"id": test['id'], "sentiment": preds})
#submission.to_csv("../data/submission_mlp.csv", index=False)

## Recurrent Neural Network (RNN)

In [11]:
# DL Pipeline 
params = {'vectorizer_method': "word2vec", 'model_name': "RNN", 'embedding_dim': 100, 
          'window': 5, 'min_count': 2, 'max_length': 300, 'hidden_dim': 128, 'num_layers': 1, 'fine_tunning': True, 
          'bidirectional': True, 'pretrained_embedding': True, 'lr': 5e-3, 'epochs': 20, 'batch_size': 256}
params['device'] = 'gpu'

pipeline = PipelineDL(**params)
pipeline

,embedding_dim,100
,window,5
,min_count,2
,max_length,300
,model_name,'RNN'
,hidden_dim,128
,num_layers,1
,fine_tunning,True
,bidirectional,True
,lr,0.005
,epochs,20


In [12]:
"""start = time.time()
pipeline.fit(X_train, y_train)
print("fit time: ", time.time()-start)

start = time.time()
score = pipeline.score(X_test, y_test)
print("prediction time: ", time.time()-start)
print(score)"""

'start = time.time()\npipeline.fit(X_train, y_train)\nprint("fit time: ", time.time()-start)\n\nstart = time.time()\nscore = pipeline.score(X_test, y_test)\nprint("prediction time: ", time.time()-start)\nprint(score)'

In [13]:
## Hyper-parameter tunning
cv = CrossValidation(pipeline=clone(pipeline))

param_grid = {
    "max_length": [10, 25, 50, 100, 200],
    "epochs": [10, 30, 50, 100]
}

#cv_rnn_results, loss_curves, _ = cv.hyper_param_tune_one_fold(X_train, y_train, X_test, y_test, param_grid, 
#                                                           reduced=False, train_size=2000, return_loss_curves=True)
#cv_rnn_results

Comentários:
- aqui conseguimos ver sintomas de duas coisas: a RNN sofre muito ao processar sequencias longas (vanishing gradient), e nas sequencias mais curtas, ele overfitta muito facil devido a alta perda de informação quanto realizamos o truncamento de cada frase.
- nas sequências mais longas, ele não consegue aprender, então não consegue nem loss boa, nem score bom.
- nas sequências mais curtas, ele consegue aprender, porém ele extrai todo o sinal possível das poucas features muito rápido, e depoid começa a overfittar com loss values ainda altos.
- para esse dataset, uma RNN não é a arquitetura ideal. A solução é tentar usar LSTMs e Transformers.

## Long Short Term Memory (LSTM)

In [14]:
# DL Pipeline 
params = {'vectorizer_method': "word2vec", 'model_name': "LSTM", 'embedding_dim': 100, 
          'window': 5, 'min_count': 2, 'max_length': 500, 'hidden_dim': 256, 'num_layers': 1, 'fine_tunning': True, 
          'bidirectional': True, 'pretrained_embedding': True, 'lr': 5e-4, 'epochs': 25, 'batch_size': 128, 'dropout': 0.3}
params['device'] = 'gpu'

pipeline = PipelineDL(**params)
pipeline

,embedding_dim,100
,window,5
,min_count,2
,max_length,500
,hidden_dim,256
,num_layers,1
,fine_tunning,True
,bidirectional,True
,dropout,0.3
,lr,0.0005
,epochs,25


In [15]:
"""start = time.time()
pipeline.fit(X_train, y_train)
print("fit time: ", time.time()-start)

start = time.time()
score = pipeline.score(X_test, y_test)
print("prediction time: ", time.time()-start)
print(score)"""

'start = time.time()\npipeline.fit(X_train, y_train)\nprint("fit time: ", time.time()-start)\n\nstart = time.time()\nscore = pipeline.score(X_test, y_test)\nprint("prediction time: ", time.time()-start)\nprint(score)'

In [17]:
## Hyper-parameter tunning
cv = CrossValidation(pipeline=clone(pipeline))

param_grid = {
    "max_length": [300],
    "hidden_dim": [128, 256],
    "lr": [1e-4, 1e-3],
    "embedding_dim": [100, 200],
    "fine_tunning": [False, True],
    "bidirectional": [False],
    "batch_size": [128],
    "dropout": [0.0, 0.2, 0.4],
    "epochs": [25]
}

cv_lstm_results, loss_curves, _ = cv.hyper_param_tune_one_fold(X_train, y_train, X_test, y_test, param_grid, 
                                                           reduced=True, train_size=10000, return_loss_curves=True)
cv_lstm_results

time list of sentences: 0.19224810600280762
Mean   : 233.7
Median : 173.0
90%    : 461
95%    : 604
99%    : 911
Max    : 1806
time word2vec: 21.122581958770752
time vocabulary: 0.015717744827270508
time embedding matrix: 0.0
time list of sentences2: 0.1322016716003418
time list of indices: 0.4509613513946533
Training Activated ? True
Model device: cuda:0
X device: cuda:0
y device: cuda:0
fc: 0.13285700976848602
rnn: 0.13200323283672333
lstm.weight_ih_l0 0.13200323283672333
lstm.weight_hh_l0 0.0274046640843153
lstm.bias_ih_l0 0.015377264469861984
lstm.bias_hh_l0 0.015377264469861984
fc.weight 0.13285700976848602
fc.bias 0.06746816635131836
Loss after Epoch 1: 0.6835.
Loss after Epoch 21: 0.3448.
Loss after gradient descent: 0.316754.
Total Training time: 102.38336992263794.
Params:  {'batch_size': 128, 'bidirectional': False, 'dropout': 0.0, 'embedding_dim': 100, 'epochs': 25, 'fine_tunning': False, 'hidden_dim': 128, 'lr': 0.0001, 'max_length': 300}
Fit time: 126.65036511421204
time p

,batch_size,bidirectional,dropout,embedding_dim,epochs,fine_tunning,hidden_dim,lr,max_length,fit_time,pred_time,final_loss,final_validation_score,score
27,128,False,0.2,200,25,False,256,0.0010,300,325.968005,4.334311,0.001965,NaN,0.8702
30,128,False,0.2,200,25,True,256,0.0001,300,380.993267,5.463962,0.143359,NaN,0.8674
41,128,False,0.4,200,25,False,128,0.0010,300,284.402797,5.818605,0.114697,NaN,0.8672
14,128,False,0.0,200,25,True,256,0.0001,300,9349.120090,5.049961,0.136355,NaN,0.8658
6,128,False,0.0,100,25,True,256,0.0001,300,331.154342,5.543104,0.187931,NaN,0.8642
10,128,False,0.0,200,25,False,256,0.0001,300,382.153110,5.458919,0.243713,NaN,0.8628
3,128,False,0.0,100,25,False,256,0.0010,300,335.596778,7.175167,0.057519,NaN,0.8628
2,128,False,0.0,100,25,False,256,0.0001,300,325.798483,9.523875,0.273709,NaN,0.8624
9,128,False,0.0,200,25,False,128,0.0010,300,293.132204,4.363507,0.166448,NaN,0.8606
1,128,False,0.0,100,25,False,128,0.0010,300,149.986307,4.295210,0.102583,NaN,0.8594


In [ ]:
## Hyper-parameter tunning
cv = CrossValidation(pipeline=clone(pipeline))

param_grid = {
    "max_length": [500],
    "hidden_dim": [256],
    "lr": [1e-3],
    "embedding_dim": [100],
    "fine_tunning": [False],
    "bidirectional": [True],
    "batch_size": [32, 64, 128],
    "dropout": [0.2],
    "epochs": [20]
}

cv_lstm_results, loss_curves, _ = cv.hyper_param_tune_one_fold(X_train, y_train, X_test, y_test, param_grid, 
                                                           reduced=False, train_size=10000, return_loss_curves=True)
cv_lstm_results

time list of sentences: 1.3548190593719482
Mean   : 233.3
Median : 174.0
90%    : 457
95%    : 596
99%    : 914
Max    : 2473


In [18]:
## Create submission using best hyper-parameters
cv.pipeline.set_params(max_length=1000)
preds = cv.pipeline.fit(X, y).predict(test)
submission = pd.DataFrame({"id": test['id'], "sentiment": preds})
submission.to_csv("../data/submission_lstm.csv", index=False)